# Day 13: SQL GROUP BY, JOINS and HAVING

---

## Setup: Import Libraries and Create Database Connection

Before we begin, we need to import `pandas` and `sqlite3`, and establish a connection to an in-memory database. We will use this database throughout the notebook.

In [1]:
import pandas as pd
import sqlite3

# Connect to an in-memory SQLite database
conn = sqlite3.connect(':memory:')

# Create a helper function to execute SQL queries and return the result as a pandas DataFrame
def run_query(query):
    return pd.read_sql_query(query, conn)


---
## Program 1: GROUP BY with COUNT

- **GROUP BY**: Used to group rows that have the same values into summary rows.
- **COUNT()**: An aggregate function that returns the number of rows that match a specified criterion.
- Together, they count the number of occurrences of each distinct value in a group.

In [2]:
# Create Student table
student_data = {
    'Roll_No': [1, 2, 3, 4, 5, 6],
    'Name': ['Alice', 'Bob', 'Charlie', 'David', 'Eva', 'Frank'],
    'Age': [20, 21, 22, 20, 21, 23],
    'Subject': ['Math', 'Science', 'Math', 'History', 'Science', 'Math']
}
df_student = pd.DataFrame(student_data)
df_student.to_sql('Student', conn, index=False, if_exists='replace')

# Execute SQL Query
query1 = """
SELECT Subject, COUNT(*) AS Student_Count
FROM Student
GROUP BY Subject;
"""

print("Output: GROUP BY with COUNT")
display(run_query(query1))

Output: GROUP BY with COUNT


,Subject,Student_Count
0,History,1
1,Math,3
2,Science,2


---
## Program 2: GROUP BY with AVG

- **AVG()**: An aggregate function that calculates the average value of a numeric column.
- Using `GROUP BY` with `AVG()` allows us to calculate the average value for each distinct group.

In [3]:
# Create Employee table
employee_data = {
    'Emp_ID': [101, 102, 103, 104, 105, 106],
    'Name': ['John', 'Sarah', 'Mike', 'Emma', 'Daniel', 'Sophia'],
    'Department': ['IT', 'HR', 'IT', 'Finance', 'HR', 'Finance'],
    'Salary': [60000, 50000, 75000, 80000, 52000, 90000]
}
df_employee = pd.DataFrame(employee_data)
df_employee.to_sql('Employee', conn, index=False, if_exists='replace')

# Execute SQL Query
query2 = """
SELECT Department,
AVG(Salary) AS Average_Salary
FROM Employee
GROUP BY Department;
"""

print("Output: GROUP BY with AVG")
display(run_query(query2))

Output: GROUP BY with AVG


,Department,Average_Salary
0,Finance,85000.0
1,HR,51000.0
2,IT,67500.0


---
## Program 3: GROUP BY with SUM

- **SUM()**: An aggregate function that returns the total sum of a numeric column.
- Combining it with `GROUP BY` helps to calculate the total sum for each category.

In [4]:
# Execute SQL Query
query3 = """
SELECT Department,
SUM(Salary) AS Total_Salary
FROM Employee
GROUP BY Department;
"""

print("Output: GROUP BY with SUM")
display(run_query(query3))

Output: GROUP BY with SUM


,Department,Total_Salary
0,Finance,170000
1,HR,102000
2,IT,135000


---
## Program 4: INNER JOIN

- **INNER JOIN**: Returns records that have matching values in both tables. If there is no match, the row is excluded.

In [5]:
# Create StudentCourse table
student_course_data = {
    'Course_ID': ['C101', 'C102', 'C103', 'C104'],
    'Roll_No': [1, 2, 4, 99] # Note: 99 is not in the Student table
}
df_student_course = pd.DataFrame(student_course_data)
df_student_course.to_sql('StudentCourse', conn, index=False, if_exists='replace')

# Execute SQL Query
query4 = """
SELECT StudentCourse.Course_ID,
Student.Name,
Student.Age
FROM Student
INNER JOIN StudentCourse
ON Student.Roll_No = StudentCourse.Roll_No;
"""

print("Output: INNER JOIN")
display(run_query(query4))

Output: INNER JOIN


,Course_ID,Name,Age
0,C101,Alice,20
1,C102,Bob,21
2,C103,David,20


---
## Program 5: LEFT JOIN

- **LEFT JOIN**: Returns all records from the left table, and the matched records from the right table. The result is NULL from the right side if there is no match.

In [6]:
# Execute SQL Query
query5 = """
SELECT Student.Name,
StudentCourse.Course_ID
FROM Student
LEFT JOIN StudentCourse
ON Student.Roll_No = StudentCourse.Roll_No;
"""

print("Output: LEFT JOIN")
display(run_query(query5))

Output: LEFT JOIN


,Name,Course_ID
0,Alice,C101
1,Bob,C102
2,Charlie,NaN
3,David,C103
4,Eva,NaN
5,Frank,NaN


---
## Program 6: RIGHT JOIN

- **RIGHT JOIN**: Returns all records from the right table, and the matched records from the left table.
- *Note*: Historically, SQLite did not support `RIGHT JOIN`. It was added in version 3.39.0. If your SQLite version is older, it would raise an error. In such cases, we can simulate a `RIGHT JOIN` by swapping the tables and using a `LEFT JOIN`.
- Assuming SQLite version supports it, we run the standard `RIGHT JOIN` syntax. Otherwise, `FROM StudentCourse LEFT JOIN Student` works.

In [7]:
# Check SQLite version
print(f"SQLite Version: {sqlite3.sqlite_version}")

# To simulate a RIGHT JOIN in older versions, we swap tables in a LEFT JOIN:
query6 = """
SELECT Student.Name,
StudentCourse.Course_ID
FROM StudentCourse
LEFT JOIN Student
ON Student.Roll_No = StudentCourse.Roll_No;
"""

print("\nOutput: RIGHT JOIN (Simulated using LEFT JOIN by swapping tables)")
display(run_query(query6))

SQLite Version: 3.50.4

Output: RIGHT JOIN (Simulated using LEFT JOIN by swapping tables)


,Name,Course_ID
0,Alice,C101
1,Bob,C102
2,David,C103
3,NaN,C104


---
## Program 7: FULL JOIN

- **FULL JOIN**: Returns all records when there is a match in either left or right table.
- *Note*: Similar to `RIGHT JOIN`, old versions of SQLite lack native `FULL OUTER JOIN`. To ensure compatibility across all versions, we simulate it using a `UNION` of a `LEFT JOIN` and a simulated `RIGHT JOIN`.

In [8]:
# Execute SQL Query
# Simulating FULL OUTER JOIN using UNION
query7 = """
SELECT Student.Name, StudentCourse.Course_ID
FROM Student
LEFT JOIN StudentCourse
ON Student.Roll_No = StudentCourse.Roll_No

UNION

SELECT Student.Name, StudentCourse.Course_ID
FROM StudentCourse
LEFT JOIN Student
ON Student.Roll_No = StudentCourse.Roll_No;
"""

print("Output: FULL JOIN (Simulated using UNION)")
display(run_query(query7))

Output: FULL JOIN (Simulated using UNION)


,Name,Course_ID
0,NaN,C104
1,Alice,C101
2,Bob,C102
3,Charlie,NaN
4,David,C103
5,Eva,NaN
6,Frank,NaN


---
## Program 8: HAVING Clause with COUNT

- **HAVING**: The `HAVING` clause was added to SQL because the `WHERE` keyword cannot be used with aggregate functions.
- It filters the grouped rows produced by the `GROUP BY` clause.

In [9]:
# Execute SQL Query
query8 = """
SELECT Department,
COUNT(*) AS Employee_Count
FROM Employee
GROUP BY Department
HAVING COUNT(*) > 1;
"""

print("Output: HAVING Clause with COUNT")
display(run_query(query8))

Output: HAVING Clause with COUNT


,Department,Employee_Count
0,Finance,2
1,HR,2
2,IT,2


---
## Program 9: HAVING Clause with AVG

- We can use the `HAVING` clause to filter out groups whose average value does not meet a specific condition.

In [10]:
# Execute SQL Query
query9 = """
SELECT Department,
AVG(Salary) AS Average_Salary
FROM Employee
GROUP BY Department
HAVING AVG(Salary) > 55000;
"""

print("Output: HAVING Clause with AVG")
display(run_query(query9))

Output: HAVING Clause with AVG


,Department,Average_Salary
0,Finance,85000.0
1,IT,67500.0


---
## Program 10: HAVING Clause with SUM

- The `HAVING` clause can filter groups based on the sum of their numeric values.

In [11]:
# Execute SQL Query
query10 = """
SELECT Department,
SUM(Salary) AS Total_Salary
FROM Employee
GROUP BY Department
HAVING SUM(Salary) > 100000;
"""

print("Output: HAVING Clause with SUM")
display(run_query(query10))

Output: HAVING Clause with SUM


,Department,Total_Salary
0,Finance,170000
1,HR,102000
2,IT,135000


---
## Final Conclusion

- **Purpose of GROUP BY**: To group rows sharing a property so that an aggregate function can be applied to them.
- **Purpose of JOINS**: To combine rows from two or more tables, based on a related column between them (e.g., INNER, LEFT, RIGHT, FULL).
- **Purpose of HAVING**: To filter groups based on conditions involving aggregate functions.
- **Difference between WHERE and HAVING**: `WHERE` filters individual rows *before* aggregation, whereas `HAVING` filters group summaries *after* aggregation.
- **Importance of SQL aggregation in data analysis**: Aggregation allows data analysts to quickly summarize massive datasets, find key metrics (totals, averages), and extract meaningful business insights.